# 07 — Baseline Multiclass ML (RF + XGBoost)

**Goal:** Train RF and XGBoost on the balanced 8-category training set.  
Evaluate both models on the **original imbalanced validation and test sets**.  
Report per-class precision, recall, F1 for all 8 categories.

**Inputs (from Google Drive):**
- `X_train_balanced_8cat_smoteenn.csv` — 1,629,654 rows (balanced)
- `y_train_balanced_8cat_smoteenn.csv` — 8-category integer labels
- `X_val.csv` — 2,100,516 rows (imbalanced, scaled)
- `y_val_multi.csv` — 34-class labels (mapped to 8 categories here)
- `X_test.csv` — 4,201,031 rows (imbalanced — NEVER TOUCHED during training)
- `y_test_multi.csv` — 34-class labels (mapped to 8 categories here)
- `label_mapping.json` — 34-class name to integer
- `category_encoding_8cat.json` — category name to integer

**Runtime:** Colab Pro — A100 GPU + High-RAM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import json, time, pickle, warnings, gc, os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import xgboost as xgb

warnings.filterwarnings('ignore')
print('All imports OK')
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_PATH   = '/content/drive/MyDrive/CICIoT2023_Research'
RESULTS_PATH = f'{DRIVE_PATH}/results'
os.makedirs(RESULTS_PATH, exist_ok=True)

# ── Category taxonomy (same as notebook 06) ────────────────────────────────────
ATTACK_TAXONOMY = {
    'DDoS':       ['DDOS-ICMP_FLOOD','DDOS-UDP_FLOOD','DDOS-TCP_FLOOD','DDOS-PSHACK_FLOOD',
                   'DDOS-SYN_FLOOD','DDOS-RSTFINFLOOD','DDOS-SYNONYMOUSIP_FLOOD',
                   'DDOS-ACK_FRAGMENTATION','DDOS-UDP_FRAGMENTATION','DDOS-ICMP_FRAGMENTATION',
                   'DDOS-SLOWLORIS','DDOS-HTTP_FLOOD'],
    'DoS':        ['DOS-UDP_FLOOD','DOS-TCP_FLOOD','DOS-SYN_FLOOD','DOS-HTTP_FLOOD'],
    'Mirai':      ['MIRAI-GREETH_FLOOD','MIRAI-UDPPLAIN','MIRAI-GREIP_FLOOD'],
    'Recon':      ['RECON-HOSTDISCOVERY','RECON-OSSCAN','RECON-PORTSCAN',
                   'RECON-PINGSWEEP','VULNERABILITYSCAN'],
    'Spoofing':   ['MITM-ARPSPOOFING','DNS_SPOOFING'],
    'Web':        ['XSS','COMMANDINJECTION','BACKDOOR_MALWARE','SQLINJECTION',
                   'UPLOADING_ATTACK','BROWSERHIJACKING'],
    'BruteForce': ['DICTIONARYBRUTEFORCE'],
    'Benign':     ['BENIGN']
}

CATEGORY_ENCODING = {cat: i for i, cat in enumerate(ATTACK_TAXONOMY.keys())}
CATEGORY_NAMES    = {v: k for k, v in CATEGORY_ENCODING.items()}
LABEL_TO_CATEGORY = {
    label: cat
    for cat, labels in ATTACK_TAXONOMY.items()
    for label in labels
}

print('Category encoding:')
for cat, idx in CATEGORY_ENCODING.items():
    print(f'  {idx}: {cat}')

## 1. Load Data

In [ ]:
# Load label mapping and build inverse (int → fine-grained name)
with open(f'{DRIVE_PATH}/label_mapping.json') as f:
    label_mapping = json.load(f)   # {name: int}

INT_TO_LABEL = {v: k for k, v in label_mapping.items()}

def map_fine_to_category(y_fine):
    """Maps 34-class integer Series → 8-category integer Series."""
    return y_fine.map(
        lambda x: CATEGORY_ENCODING[LABEL_TO_CATEGORY[INT_TO_LABEL[x]]]
    )

print(f'Label mapping loaded — {len(label_mapping)} fine-grained classes.')

In [ ]:
# Load X_val first to get feature names
# (X_train_balanced was saved with integer column names after SMOTE-ENN)
print('Loading X_val (to get feature names)...')
t0 = time.time()
X_val = pd.read_csv(f'{DRIVE_PATH}/X_val.csv')
FEATURE_NAMES = X_val.columns.tolist()
print(f'X_val: {X_val.shape}  ({time.time()-t0:.1f}s)')

In [ ]:
# Load balanced training data
print('Loading balanced training data...')
t0 = time.time()
X_train = pd.read_csv(f'{DRIVE_PATH}/X_train_balanced_8cat_smoteenn.csv')
X_train.columns = FEATURE_NAMES   # Restore real feature names

y_train = pd.read_csv(
    f'{DRIVE_PATH}/y_train_balanced_8cat_smoteenn.csv', header=0
).squeeze()

print(f'X_train: {X_train.shape}  ({time.time()-t0:.1f}s)')
print('y_train distribution (balanced):')
for cat_int, count in sorted(y_train.value_counts().items()):
    print(f'  {cat_int} {CATEGORY_NAMES[cat_int]:<12}: {count:>8,}')

In [ ]:
# Load and map val labels (34-class → 8-category)
y_val_fine = pd.read_csv(
    f'{DRIVE_PATH}/y_val_multi.csv', header=0, dtype=int
).squeeze()
y_val = map_fine_to_category(y_val_fine)

print('y_val distribution (imbalanced):')
for cat_int, count in sorted(y_val.value_counts().items()):
    pct = count / len(y_val) * 100
    print(f'  {cat_int} {CATEGORY_NAMES[cat_int]:<12}: {count:>8,}  ({pct:.2f}%)')

In [ ]:
# Load test data — NEVER used during training/tuning
print('Loading X_test and y_test (imbalanced — final evaluation only)...')
t0 = time.time()
X_test = pd.read_csv(f'{DRIVE_PATH}/X_test.csv')

y_test_fine = pd.read_csv(
    f'{DRIVE_PATH}/y_test_multi.csv', header=0, dtype=int
).squeeze()
y_test = map_fine_to_category(y_test_fine)

print(f'X_test: {X_test.shape}  ({time.time()-t0:.1f}s)')
print('y_test distribution (imbalanced — original):')
for cat_int, count in sorted(y_test.value_counts().items()):
    pct = count / len(y_test) * 100
    print(f'  {cat_int} {CATEGORY_NAMES[cat_int]:<12}: {count:>8,}  ({pct:.2f}%)')

In [ ]:
# Sanity checks
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 39
assert len(X_train) == len(y_train)
assert len(X_val)   == len(y_val)
assert len(X_test)  == len(y_test)
assert set(y_train.unique()) == set(range(8))
assert set(y_test.unique())  == set(range(8))
print('All sanity checks passed.')
print(f'  Train: {len(X_train):,} rows (balanced)')
print(f'  Val:   {len(X_val):,} rows (imbalanced)')
print(f'  Test:  {len(X_test):,} rows (imbalanced)')

## 2. Helper Functions

In [ ]:
def evaluate_model(model, X, y, split_name, model_name):
    """Evaluate model, print per-class report, return metrics dict."""
    y_pred = model.predict(X)
    acc    = accuracy_score(y, y_pred)
    f1_mac = f1_score(y, y_pred, average='macro',    zero_division=0)
    f1_wt  = f1_score(y, y_pred, average='weighted', zero_division=0)
    target_names = [CATEGORY_NAMES[i] for i in sorted(CATEGORY_NAMES)]

    print(f'\n{"="*60}')
    print(f'{model_name} — {split_name}')
    print(f'{"="*60}')
    print(f'  Accuracy:       {acc:.4f}')
    print(f'  F1 (macro):     {f1_mac:.4f}')
    print(f'  F1 (weighted):  {f1_wt:.4f}')
    print()
    print(classification_report(y, y_pred, target_names=target_names, zero_division=0))

    report = classification_report(
        y, y_pred, target_names=target_names, output_dict=True, zero_division=0
    )
    return {
        'model': model_name, 'split': split_name,
        'accuracy': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_wt,
        'per_class': {
            name: {
                'precision': report[name]['precision'],
                'recall':    report[name]['recall'],
                'f1':        report[name]['f1-score'],
                'support':   int(report[name]['support'])
            } for name in target_names
        },
        'y_pred': y_pred
    }


def plot_confusion_matrix(y_true, y_pred, model_name, split_name, save_path=None):
    """Normalized confusion matrix (rows = true labels = recall per class)."""
    labels = [CATEGORY_NAMES[i] for i in sorted(CATEGORY_NAMES)]
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        vmin=0, vmax=1, ax=ax, linewidths=0.5
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'{model_name} — Confusion Matrix (Normalized)\n{split_name}', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


def plot_feature_importance(model, feature_names, model_name, top_n=20, save_path=None):
    """Top N feature importances horizontal bar chart."""
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(top_n), importances[indices][::-1],
            align='center', color='steelblue', edgecolor='white')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in indices[::-1]], fontsize=10)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'{model_name} — Top {top_n} Feature Importances', fontsize=13)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()
    print(f'\nTop {top_n} features ({model_name}):')
    for rank, idx in enumerate(indices, 1):
        print(f'  {rank:2d}. {feature_names[idx]:<25} {importances[idx]:.4f}')


print('Helper functions defined.')

## 3. Random Forest

RF is CPU-only — the A100 GPU is not used here.  
`n_jobs=-1` uses all available CPU cores on the High-RAM instance (~12-16 cores).  
Expected training time: **15–30 minutes** on 1.6M rows.

In [ ]:
RF_PARAMS = dict(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight=None,   # Training data already balanced
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print('Training Random Forest...')
print(f'  Params: {RF_PARAMS}')
t0 = time.time()

rf = RandomForestClassifier(**RF_PARAMS)
rf.fit(X_train, y_train)

rf_train_time = time.time() - t0
print(f'\nRF training complete: {rf_train_time:.1f}s ({rf_train_time/60:.1f} min)')

In [ ]:
# Evaluate RF on Validation
rf_val = evaluate_model(rf, X_val, y_val, 'Validation', 'Random Forest')

In [ ]:
# Evaluate RF on Test (the real evaluation — imbalanced)
rf_test = evaluate_model(rf, X_test, y_test, 'Test (Imbalanced)', 'Random Forest')

In [ ]:
plot_confusion_matrix(
    y_test, rf_test['y_pred'], 'Random Forest', 'Test Set (Imbalanced)',
    save_path=f'{RESULTS_PATH}/rf_confusion_matrix_test.png'
)

In [ ]:
plot_feature_importance(
    rf, FEATURE_NAMES, 'Random Forest', top_n=20,
    save_path=f'{RESULTS_PATH}/rf_feature_importance.png'
)

In [ ]:
# Save RF model
with open(f'{DRIVE_PATH}/rf_model_8cat_baseline.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('RF model saved.')

# Stash predictions, free memory
rf_test_pred = rf_test['y_pred'].copy()
rf_val_pred  = rf_val['y_pred'].copy()
del rf_test['y_pred'], rf_val['y_pred']
gc.collect()
print('Memory freed.')

## 4. XGBoost

XGBoost can use the A100 GPU via `tree_method='hist'` + `device='cuda'`.  
Expected training time on A100: **3–8 minutes** for 300 trees on 1.6M rows.

In [ ]:
XGB_PARAMS = dict(
    objective='multi:softprob',
    num_class=8,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',   # Required for GPU
    device='cuda',        # Use A100
    eval_metric='mlogloss',
    random_state=42,
    verbosity=1
)

print('Training XGBoost (GPU — A100)...')
print(f'  Params: {XGB_PARAMS}')
t0 = time.time()

xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

xgb_train_time = time.time() - t0
print(f'\nXGBoost training complete: {xgb_train_time:.1f}s ({xgb_train_time/60:.1f} min)')

In [ ]:
xgb_val  = evaluate_model(xgb_model, X_val,  y_val,  'Validation',        'XGBoost')

In [ ]:
xgb_test = evaluate_model(xgb_model, X_test, y_test, 'Test (Imbalanced)', 'XGBoost')

In [ ]:
plot_confusion_matrix(
    y_test, xgb_test['y_pred'], 'XGBoost', 'Test Set (Imbalanced)',
    save_path=f'{RESULTS_PATH}/xgb_confusion_matrix_test.png'
)

In [ ]:
plot_feature_importance(
    xgb_model, FEATURE_NAMES, 'XGBoost', top_n=20,
    save_path=f'{RESULTS_PATH}/xgb_feature_importance.png'
)

In [ ]:
with open(f'{DRIVE_PATH}/xgb_model_8cat_baseline.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
print('XGBoost model saved.')

xgb_test_pred = xgb_test['y_pred'].copy()
xgb_val_pred  = xgb_val['y_pred'].copy()
del xgb_test['y_pred'], xgb_val['y_pred']
gc.collect()

## 5. Comparison and Final Results

In [ ]:
print('\n' + '='*70)
print('FINAL RESULTS — TEST SET (IMBALANCED)')
print('='*70)
print(f'{"":<16} {"RF":>10} {"XGBoost":>10}')
print('-'*38)
print(f'{"Accuracy":<16} {rf_test["accuracy"]:>10.4f} {xgb_test["accuracy"]:>10.4f}')
print(f'{"F1 macro":<16} {rf_test["f1_macro"]:>10.4f} {xgb_test["f1_macro"]:>10.4f}')
print(f'{"F1 weighted":<16} {rf_test["f1_weighted"]:>10.4f} {xgb_test["f1_weighted"]:>10.4f}')
print(f'{"Train time":<16} {rf_train_time/60:>9.1f}m {xgb_train_time/60:>9.1f}m')
print()

print('PER-CLASS F1 — TEST SET')
print(f'{"Category":<13} {"RF Pre":>7} {"RF Rec":>7} {"RF F1":>7} | {"XGB Pre":>8} {"XGB Rec":>8} {"XGB F1":>8}  {"Support":>9}')
print('-'*80)
for cat_int in sorted(CATEGORY_NAMES):
    name = CATEGORY_NAMES[cat_int]
    r  = rf_test['per_class'][name]
    x  = xgb_test['per_class'][name]
    print(f'{name:<13} {r["precision"]:>7.4f} {r["recall"]:>7.4f} {r["f1"]:>7.4f} |'
          f' {x["precision"]:>8.4f} {x["recall"]:>8.4f} {x["f1"]:>8.4f}  {r["support"]:>9,}')

In [ ]:
# Side-by-side F1 bar chart
categories = [CATEGORY_NAMES[i] for i in sorted(CATEGORY_NAMES)]
rf_f1s     = [rf_test['per_class'][c]['f1']  for c in categories]
xgb_f1s    = [xgb_test['per_class'][c]['f1'] for c in categories]

x = np.arange(len(categories))
w = 0.35
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - w/2, rf_f1s,  w, label='Random Forest', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, xgb_f1s, w, label='XGBoost',       color='coral',     alpha=0.85)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title(
    'Per-Class F1 — RF vs XGBoost\n'
    'Trained on balanced data | Evaluated on imbalanced test set',
    fontsize=13
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_path = f'{RESULTS_PATH}/ml_baseline_f1_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved: {save_path}')
plt.show()

In [ ]:
# Save results CSV
rows = []
for cat_int in sorted(CATEGORY_NAMES):
    name = CATEGORY_NAMES[cat_int]
    r = rf_test['per_class'][name]
    x = xgb_test['per_class'][name]
    rows.append({
        'category': name, 'category_int': cat_int, 'support': r['support'],
        'rf_precision': r['precision'], 'rf_recall': r['recall'], 'rf_f1': r['f1'],
        'xgb_precision': x['precision'], 'xgb_recall': x['recall'], 'xgb_f1': x['f1'],
    })

for label, rf_v, xgb_v in [
    ('ACCURACY', rf_test['accuracy'], xgb_test['accuracy']),
    ('F1_MACRO', rf_test['f1_macro'], xgb_test['f1_macro']),
    ('F1_WEIGHTED', rf_test['f1_weighted'], xgb_test['f1_weighted']),
]:
    rows.append({
        'category': label, 'category_int': -1, 'support': len(y_test),
        'rf_precision': rf_v,  'rf_recall': rf_v,  'rf_f1': rf_v,
        'xgb_precision': xgb_v, 'xgb_recall': xgb_v, 'xgb_f1': xgb_v,
    })

results_df = pd.DataFrame(rows)
csv_path   = f'{RESULTS_PATH}/results_ml_baseline_8cat_imbalanced_test.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results saved: {csv_path}')
results_df

In [ ]:
# Save experiment log
experiment_log = {
    'notebook':          '07_Baseline_Multiclass_ML',
    'dataset':           'CICIoT2023',
    'train_rows':        int(len(X_train)),
    'val_rows':          int(len(X_val)),
    'test_rows':         int(len(X_test)),
    'features':          39,
    'num_classes':       8,
    'train_balanced':    True,
    'test_imbalanced':   True,
    'balancing_method':  'RandomUnderSampler + SMOTE-ENN (from nb06)',
    'rf': {
        'params':        str(RF_PARAMS),
        'train_time_s':  round(rf_train_time, 1),
        'test_accuracy': round(rf_test['accuracy'],   4),
        'test_f1_macro': round(rf_test['f1_macro'],   4),
        'test_f1_wt':    round(rf_test['f1_weighted'], 4),
    },
    'xgboost': {
        'params':        str(XGB_PARAMS),
        'train_time_s':  round(xgb_train_time, 1),
        'test_accuracy': round(xgb_test['accuracy'],   4),
        'test_f1_macro': round(xgb_test['f1_macro'],   4),
        'test_f1_wt':    round(xgb_test['f1_weighted'], 4),
    },
    'notes': [
        'Test set uses original imbalanced distribution — never touched during training',
        'Training set balanced with SMOTE-ENN + RandomUnderSampler',
        'XGBoost trained with A100 GPU (device=cuda, tree_method=hist)',
        'RF trained on CPU (n_jobs=-1)',
        'Per-class F1 on imbalanced test is the key metric — not overall accuracy'
    ]
}

log_path = f'{RESULTS_PATH}/experiment_log_07_ml_baseline.json'
with open(log_path, 'w') as f:
    json.dump(experiment_log, f, indent=2)
print(f'Experiment log saved: {log_path}')

## 6. What to Look For in the Results

| Category | Expected F1 | Why |
|---|---|---|
| DDoS, DoS, Mirai | > 0.95 | Large training samples, very distinct traffic signatures (low IAT, high SYN) |
| Benign | > 0.90 | Well-represented, distinctive from attacks |
| Recon, Spoofing | 0.70 – 0.90 | Some overlap with normal scanning/ARP traffic |
| Web, BruteForce | Variable | Hardest: smallest classes, patterns can resemble legitimate web traffic |

**Key questions after running:**
1. What is the Web and BruteForce F1? This is the most important number — it shows whether SMOTE-ENN worked.
2. Is there a big gap between val F1 and test F1 for minority classes? Expected — test is imbalanced.
3. Which features dominate importance? IAT, syn_flag_number, Rate should rank highest.
4. Does XGBoost outperform RF on minority classes, or does RF do better?

**These results are your Stage 1 upper bound.** Every FL experiment in Stage 2 is compared against these numbers.

In [ ]:
print('\n' + '='*60)
print('NOTEBOOK 07 COMPLETE')
print('='*60)
print(f'\nSaved to {DRIVE_PATH}/')
print('  rf_model_8cat_baseline.pkl')
print('  xgb_model_8cat_baseline.pkl')
print(f'\nSaved to {RESULTS_PATH}/')
print('  results_ml_baseline_8cat_imbalanced_test.csv')
print('  experiment_log_07_ml_baseline.json')
print('  rf_confusion_matrix_test.png')
print('  xgb_confusion_matrix_test.png')
print('  rf_feature_importance.png')
print('  xgb_feature_importance.png')
print('  ml_baseline_f1_comparison.png')
print('\nNext: 08_Baseline_Multiclass_DL.ipynb (MLP + 1D CNN)')